# Manipulation et analyse d'images avec `scikit-image`

Dans ce TP, vous allez utiliser quelques-unes des fonctionnalités de scikit-image pour manipuler et analyser une photo : corriger son contraste, en détecter les contours, détecter et apparier des points caractéristiques, recoller deux images qui se chevauchent, puis comparer des images par des métriques. Sur une autre image, des pièces de monnaie, vous compterez et mesurerez aussi des objets.

1. Histogramme d'une image
2. Corriger le contraste
3. Détecter les contours, avant et après un flou
4. Seuiller, compter et mesurer des objets
5. Détecter des points caractéristiques
6. Apparier les points caractéristiques
7. Recoller deux images qui se chevauchent
8. Métriques de comparaison d'images

scikit-image travaille sur des tableaux NumPy : il faut être à l'aise avec leur manipulation (indexation, tranches, opérations sur des tableaux entiers) pour en tirer parti.

Les exercices s'enchaînent : chacun se sert de ce que les solutions des précédents définissent. Chaque exercice a une cellule pour votre code, puis des conseils supplémentaires et sa solution, repliés : ouvrez-les si besoin.

Commencez par exécuter la cellule suivante, sans la lire : elle télécharge la photo du TP et définit la fonction `cote_a_cote`, qui affiche des images côte à côte.

In [ ]:
# @title À exécuter : mise en place de l'environnement
# La photo du TP : Ezarate, CC BY-SA 4.0, Wikimedia Commons.
!curl -fsSL -o la-havana.jpg https://upload.wikimedia.org/wikipedia/commons/thumb/7/74/AvMalecon-LaHabanaCuba-04735.jpg/1920px-AvMalecon-LaHabanaCuba-04735.jpg

from collections.abc import Sequence

import matplotlib.pyplot as plt
import numpy as np
from skimage import (
  color,
  data,
  exposure,
  feature,
  filters,
  io,
  measure,
  metrics,
  transform,
  util,
)


def cote_a_cote(
  images: Sequence[np.ndarray],
  titre: str | None = None,
  sous_titres: Sequence[str] | None = None,
  dpi: int = 150,
) -> None:
  """Affiche des images côte à côte, chacune avec son sous-titre."""
  fig, axes = plt.subplots(1, len(images), dpi=dpi)
  for i, (ax, image) in enumerate(zip(axes, images)):
    # Une image à un seul canal s'affiche en niveaux de gris.
    ax.imshow(image, cmap="gray" if image.ndim == 2 else None)
    ax.axis("off")
    if sous_titres:
      ax.set_title(sous_titres[i], fontsize=8)
  if titre:
    fig.suptitle(titre, y=0.85)
  fig.tight_layout()
  plt.show()

## 1. Histogramme d'une image

La photo du TP, `la-havana.jpg`, que la cellule de mise en place a téléchargée, montre l'avenue du Malecón, à La Havane (photo d'Ezarate, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/deed.fr), [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:AvMalecon-LaHabanaCuba-04735.jpg)).

L'histogramme d'une image compte ses pixels par intensité, de 0 à 255 pour une image `uint8` : il montre d'un coup d'œil si l'image est sombre, claire ou peu contrastée, et les corrections de contraste de l'exercice suivant reposent sur lui.

- Chargez la photo dans une variable `im` avec `io.imread`, comme dans le cours, et affichez sa forme et son type.
- Écrivez la fonction `afficher_histogramme(image, cumule=False)`, qui calcule l'histogramme de chacun des trois canaux de `image` avec `exposure.histogram` et trace les trois courbes sur un même graphique, chacune de la couleur de son canal. Avec `cumule=True`, elle trace l'histogramme cumulé de chaque canal : pour chaque intensité, le nombre de pixels d'intensité inférieure ou égale.
- Affichez l'histogramme de `im`, puis son histogramme cumulé. Les exercices suivants se servent de `afficher_histogramme` pour comparer les histogrammes avant et après une correction.

**Documentation utile**

- https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.histogram
- https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html

In [ ]:
def afficher_histogramme(image: np.ndarray, cumule: bool = False) -> None:
  # Votre code ici
  ...


# Votre code ici

### Conseils supplémentaires

- `io.imread("la-havana.jpg")` renvoie un tableau NumPy `uint8` de forme `(hauteur, largeur, canaux)` : vous devez obtenir `(1247, 1920, 3)`.
- `exposure.histogram(image, channel_axis=2)` calcule un histogramme par canal, les canaux étant sur l'axe 2. Il renvoie un couple `(histogrammes, valeurs)` : `histogrammes` a une ligne par canal, `(3, 256)` pour `im`, et `valeurs` donne les intensités, en abscisse.
- `np.cumsum(histogramme)` donne l'histogramme cumulé. Sa dernière valeur est le nombre de pixels de l'image, 1247 × 1920 = 2 394 240.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def afficher_histogramme(image: np.ndarray, cumule: bool = False) -> None:
  histogrammes, valeurs = exposure.histogram(image, channel_axis=...)
  for histogramme, couleur in zip(histogrammes, ("red", "green", "blue")):
    if cumule:
      histogramme = ...
    plt.plot(valeurs, histogramme, color=couleur)
  plt.xlabel("Intensité")
  plt.ylabel("Nombre de pixels")
  plt.show()


im = io.imread(...)
print(im.shape, im.dtype)
afficher_histogramme(im)
afficher_histogramme(im, cumule=True)
```

### Solution

In [ ]:
def afficher_histogramme(image: np.ndarray, cumule: bool = False) -> None:
  histogrammes, valeurs = exposure.histogram(image, channel_axis=2)
  for histogramme, couleur in zip(histogrammes, ("red", "green", "blue")):
    if cumule:
      histogramme = np.cumsum(histogramme)
    plt.plot(valeurs, histogramme, color=couleur)
  plt.xlabel("Intensité")
  plt.ylabel("Nombre de pixels")
  plt.show()


im = io.imread("la-havana.jpg")
print(im.shape, im.dtype)
afficher_histogramme(im)
afficher_histogramme(im, cumule=True)

## 2. Corriger le contraste

Le module `exposure` fournit plusieurs corrections de contraste fondées sur l'histogramme. Appliquez-en trois à `im` ; affichez chaque résultat à côté de l'original avec `cote_a_cote`, puis son histogramme et son histogramme cumulé.

- `exposure.equalize_hist` égalise l'histogramme de toute l'image, une correction non linéaire : chaque intensité est remplacée par la part des pixels d'intensité inférieure ou égale, ce qui rapproche l'histogramme cumulé d'une droite. C'est l'égalisation que le TP Pillow et NumPy écrit à la main, sur une image en niveaux de gris : comparez l'histogramme cumulé avant et après. Sur une image en couleur, la fonction affiche un avertissement (`UserWarning: This might be a color image…`) : elle calcule un seul histogramme, sur les valeurs des trois canaux mises ensemble, et applique la même transformation aux trois, ce qui peut décaler les couleurs.
- `exposure.equalize_adapthist` égalise l'histogramme région par région (CLAHE, *Contrast Limited Adaptive Histogram Equalization*) : le contraste s'adapte à chaque partie de l'image.
- `exposure.rescale_intensity` étire linéairement un intervalle d'intensités sur toute la plage. Choisissez l'intervalle avec `np.percentile` : de l'intensité sous laquelle se trouvent 2 % des pixels à celle sous laquelle s'en trouvent 98 %.

Quelle correction rend le mieux le ciel et ses nuages ? La chaussée ? Laquelle garde les couleurs de la voiture ?

**Documentation utile**

- https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.equalize_hist
- https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.equalize_adapthist
- https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.rescale_intensity
- https://numpy.org/doc/stable/reference/generated/numpy.percentile.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `exposure.equalize_hist(im)` et `exposure.equalize_adapthist(im)` renvoient des flottants entre 0 et 1, que `imshow` et `afficher_histogramme` acceptent tels quels.
- `np.percentile(im, (2, 98))` renvoie les deux bornes de l'intervalle, 41 et 233 pour cette photo.
- `exposure.rescale_intensity(im, in_range=(bas, haut))` envoie `bas` sur la plus petite valeur du type de l'image (0 pour `uint8`) et `haut` sur la plus grande (255) ; les intensités en dehors de l'intervalle sont ramenées à ces bornes.
- `cote_a_cote((im, im_egalisee), sous_titres=("Originale", "Égalisée"))` affiche deux images côte à côte.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
im_egalisee = exposure.equalize_hist(im)
cote_a_cote((im, im_egalisee), sous_titres=("Originale", "Égalisée"))
afficher_histogramme(im_egalisee)
afficher_histogramme(im_egalisee, cumule=True)

im_adaptative = ...
...

bas, haut = np.percentile(im, ...)
im_etiree = exposure.rescale_intensity(im, in_range=...)
...
```

### Solution

In [ ]:
im_egalisee = exposure.equalize_hist(im)
cote_a_cote((im, im_egalisee), sous_titres=("Originale", "Égalisée"))
afficher_histogramme(im_egalisee)
afficher_histogramme(im_egalisee, cumule=True)

In [ ]:
im_adaptative = exposure.equalize_adapthist(im)
cote_a_cote((im, im_adaptative), sous_titres=("Originale", "Égalisation adaptative"))
afficher_histogramme(im_adaptative)
afficher_histogramme(im_adaptative, cumule=True)

In [ ]:
# Les intensités entre le 2e et le 98e centile, étirées sur toute la plage.
bas, haut = np.percentile(im, (2, 98))
im_etiree = exposure.rescale_intensity(im, in_range=(bas, haut))
cote_a_cote((im, im_etiree), sous_titres=("Originale", "Intensités étirées"))
afficher_histogramme(im_etiree)
afficher_histogramme(im_etiree, cumule=True)

# L'égalisation globale fonce le ciel et détache les nuages, mais assombrit la
# chaussée et décale les couleurs : la voiture bleuit, le trottoir jaunit.
# L'égalisation adaptative fait ressortir le relief des nuages et le grain de la
# chaussée, sans changer les couleurs. L'étirement linéaire, plus doux, garde
# des couleurs naturelles.

## 3. Détecter les contours, avant et après un flou

Le filtre de Sobel mesure en chaque pixel la variation d'intensité autour de lui : grande sur un contour, nulle dans une zone uniforme. Mais le grain d'une photo, celui de l'asphalte par exemple, est fait lui aussi de petites variations, que Sobel prend pour des contours. Un flou gaussien les efface avant la détection.

- Convertissez `im` en niveaux de gris dans `im_gris`, et appliquez-lui `filters.sobel`, dans `contours_bruts`.
- Floutez `im_gris` avec `filters.gaussian`, d'écart type `sigma=2`, dans `im_floue`, puis appliquez `filters.sobel` à l'image floutée, dans `contours_flous`.
- Affichez les deux résultats côte à côte avec `cote_a_cote`, puis un détail des deux, la voiture : les lignes 800 et suivantes, les colonnes 450 à 1200.
- Mesurez le bruit des contours : la moyenne de chacun des deux résultats sur la chaussée, les 100 dernières lignes de l'image, où presque tout ce que Sobel trouve est le grain de l'asphalte.

Quels contours le flou efface-t-il ? Lesquels garde-t-il, et que leur fait-il ? Recommencez avec `sigma=1`, puis avec `sigma=4` : que gagne-t-on, que perd-on ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `color.rgb2gray(im)` renvoie des flottants entre 0 et 1, de forme `(1247, 1920)` ; `filters.gaussian` et `filters.sobel` renvoient eux aussi des flottants.
- Plus `sigma` est grand, plus le flou est fort.
- `contours_bruts[800:, 450:1200]` est le détail de la voiture, `contours_bruts[-100:]` les 100 dernières lignes, et `.mean()` donne la moyenne d'un tableau. Vous devez trouver environ 0,076 sans flou, et 0,008 avec : le flou divise le bruit par dix.
- `cote_a_cote` affiche une image à un seul canal en niveaux de gris, sa plus forte valeur en blanc.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
im_gris = color.rgb2gray(im)
contours_bruts = filters.sobel(im_gris)
im_floue = filters.gaussian(im_gris, sigma=...)
contours_flous = ...

sous_titres = ("Sobel", "Flou gaussien, puis Sobel")
cote_a_cote((contours_bruts, contours_flous), sous_titres=sous_titres)
cote_a_cote((contours_bruts[800:, 450:1200], ...), sous_titres=sous_titres)
print(contours_bruts[...].mean(), contours_flous[...].mean())
```

### Solution

In [ ]:
im_gris = color.rgb2gray(im)
contours_bruts = filters.sobel(im_gris)
# Le flou efface le grain avant la détection des contours.
im_floue = filters.gaussian(im_gris, sigma=2)
contours_flous = filters.sobel(im_floue)

sous_titres = ("Sobel", "Flou gaussien, puis Sobel")
cote_a_cote((contours_bruts, contours_flous), sous_titres=sous_titres)
# Un détail : la voiture.
cote_a_cote(
  (contours_bruts[800:, 450:1200], contours_flous[800:, 450:1200]),
  sous_titres=sous_titres,
)
# Le bruit des contours sur la chaussée, les 100 dernières lignes.
print(contours_bruts[-100:].mean(), contours_flous[-100:].mean())

# Sans flou, la chaussée se couvre de petits contours, ceux du grain de
# l'asphalte. Après le flou, il n'en reste presque rien (une moyenne dix fois
# plus faible), alors que les contours nets, ceux de la carrosserie, des vitres
# et des réverbères, restent, plus épais et moins précis. Avec sigma=1, une
# partie du grain reste ; avec sigma=4, il disparaît, mais les détails fins
# aussi : la plaque d'immatriculation et les jantes s'effacent.

## 4. Seuiller, compter et mesurer des objets

Compter des objets et les mesurer est une analyse d'image courante : des cellules, des grains, des pièces. La chaîne la plus simple a trois étapes : un seuillage sépare les objets du fond, l'étiquetage numérote les régions d'un seul tenant du masque obtenu, puis chaque région est mesurée. La photo du Malecón ne s'y prête pas : cet exercice prend l'image `data.coins()`, livrée avec scikit-image, qui montre 24 pièces de monnaie grecques sur un fond gris.

- Chargez l'image dans `pieces`.
- Calculez son seuil d'Otsu, `seuil`, avec `filters.threshold_otsu`, puis le masque `masque`, vrai pour les pixels plus clairs que le seuil. Affichez l'image et le masque côte à côte.
- Étiquetez les régions du masque avec `measure.label`, dans `etiquettes`, et affichez le nombre de régions trouvées.
- Mesurez les régions avec `measure.regionprops`, et gardez dans `grandes` celles d'au moins 100 pixels d'aire (`area`) : les autres sont des taches du fond. Affichez leur nombre.
- Affichez l'image avec, sur chaque grande région, une croix à son centre (`centroid`) et son aire.

Trouvez-vous autant de grandes régions que de pièces ? Chaque région est-elle une pièce ? Regardez la plus grande.

**Documentation utile**

- https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.coins
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.text.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `data.coins()` renvoie une image en niveaux de gris, un tableau `uint8` de forme `(303, 384)`.
- `filters.threshold_otsu(pieces)` renvoie le seuil, 107 ici, et `pieces > seuil` le masque, un tableau de booléens.
- `measure.label(masque)` donne à chaque pixel du masque le numéro de sa région, de 1 au nombre de régions, et 0 au fond : `etiquettes.max()` est le nombre de régions, 96.
- `measure.regionprops(etiquettes)` renvoie une liste d'objets, un par région, avec leurs mesures en attributs : `area`, l'aire en pixels, et `centroid`, le centre, un couple `(ligne, colonne)`. Vous devez garder 24 régions.
- `plt.scatter(colonne, ligne, marker="+")` et `plt.text(colonne, ligne, texte)` placent une croix et un texte : l'abscisse, la colonne, avant l'ordonnée, la ligne.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
pieces = data.coins()
seuil = filters.threshold_otsu(...)
masque = ...
cote_a_cote((pieces, masque), sous_titres=(f"Seuil d'Otsu : {seuil}", "Masque"))

etiquettes = measure.label(...)
print(etiquettes.max(), "régions")

grandes = [region for region in measure.regionprops(etiquettes) if ...]
print(len(grandes), "régions d'au moins 100 pixels")

plt.figure(dpi=150)
plt.imshow(pieces, cmap="gray")
for region in grandes:
  ligne, colonne = region.centroid
  plt.scatter(..., ..., marker="+", color="red")
  plt.text(colonne, ligne, str(int(region.area)), color="yellow", fontsize=6)
plt.axis("off")
plt.show()
```

### Solution

In [ ]:
pieces = data.coins()
seuil = filters.threshold_otsu(pieces)
# Vrai pour les pixels plus clairs que le seuil : les pièces, sur un fond sombre.
masque = pieces > seuil
cote_a_cote((pieces, masque), sous_titres=(f"Seuil d'Otsu : {seuil}", "Masque"))

# Un numéro par région d'un seul tenant, 0 pour le fond.
etiquettes = measure.label(masque)
print(etiquettes.max(), "régions")

# Les petites régions sont des taches du fond plus claires que le seuil.
grandes = [region for region in measure.regionprops(etiquettes) if region.area >= 100]
print(len(grandes), "régions d'au moins 100 pixels")

plt.figure(dpi=150)
plt.imshow(pieces, cmap="gray")
for region in grandes:
  # Le centre est un couple (ligne, colonne) : la colonne est l'abscisse.
  ligne, colonne = region.centroid
  plt.scatter(colonne, ligne, marker="+", color="red")
  plt.text(colonne, ligne, str(int(region.area)), color="yellow", fontsize=6)
plt.axis("off")
plt.show()

# 24 grandes régions pour 24 pièces : le compte tombe juste, mais par hasard.
# La pièce en haut à gauche touche une bande claire du bord supérieur, et forme
# avec elle la plus grande région (8792 pixels), dont le centre ne tombe sur
# aucune pièce : 23 régions sont des pièces seules, la 24e mêle une pièce et
# la bande. Les aires varient du simple au triple : les pièces n'ont pas toutes
# la même taille, et le seuil troue les plus sombres. On vérifie toujours le résultat
# sur l'image, pas seulement le nombre d'objets.

## 5. Détecter des points caractéristiques

Un détecteur de points caractéristiques repère dans une image des points remarquables, comme des coins, et décrit le voisinage de chacun par un descripteur, une signature qui se compare d'une image à l'autre. ORB (*Oriented FAST and Rotated BRIEF*) en est un, conçu pour retrouver les mêmes points dans une image tournée.

- Tournez `im` de 45° avec `transform.rotate`, comme dans le cours, dans une nouvelle image `im_tournee`, et affichez les deux images côte à côte.
- Créez un détecteur `feature.ORB`, et appelez sa méthode `detect_and_extract` sur `im` en niveaux de gris : le détecteur range alors les points trouvés dans son attribut `keypoints`, et leurs descripteurs dans `descriptors`. Gardez-les dans `points_1` et `descripteurs_1`.
- Faites de même avec `im_tournee`, dans `points_2` et `descripteurs_2`.
- Affichez chaque image avec ses points par-dessus.

**Documentation utile**

- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.ORB
- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.ORB.detect_and_extract
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `detect_and_extract` attend une image à un seul canal : `color.rgb2gray(im)`.
- Le même détecteur sert aux deux images : ses attributs `keypoints` et `descriptors` sont remplacés à chaque appel, lisez-les avant l'appel suivant.
- `keypoints` est un tableau de forme `(500, 2)` : ORB garde 500 points par défaut, chacun un couple `(ligne, colonne)`. `descriptors` a une ligne de 256 booléens par point.
- `plt.scatter(x, y)` attend l'abscisse, la colonne, avant l'ordonnée, la ligne : `points[:, 1]`, puis `points[:, 0]`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
im_tournee = transform.rotate(im, ...)
cote_a_cote((im, im_tournee), sous_titres=("Originale", "Tournée de 45°"))

detecteur = feature.ORB()
detecteur.detect_and_extract(color.rgb2gray(im))
points_1 = detecteur.keypoints
descripteurs_1 = ...

detecteur.detect_and_extract(...)
points_2 = ...
descripteurs_2 = ...

for image, points in ((im, points_1), (im_tournee, points_2)):
  plt.imshow(image)
  plt.scatter(..., ..., s=5)
  plt.axis("off")
  plt.show()
```

### Solution

In [ ]:
im_tournee = transform.rotate(im, 45)
cote_a_cote((im, im_tournee), sous_titres=("Originale", "Tournée de 45°"))

In [ ]:
detecteur = feature.ORB()

# Les points et les descripteurs de l'image, en niveaux de gris.
detecteur.detect_and_extract(color.rgb2gray(im))
points_1 = detecteur.keypoints
descripteurs_1 = detecteur.descriptors

# Le détecteur remplace ses attributs à chaque appel.
detecteur.detect_and_extract(color.rgb2gray(im_tournee))
points_2 = detecteur.keypoints
descripteurs_2 = detecteur.descriptors

for image, points in ((im, points_1), (im_tournee, points_2)):
  plt.imshow(image)
  # Les points sont des couples (ligne, colonne) : la colonne est l'abscisse.
  plt.scatter(points[:, 1], points[:, 0], s=5)
  plt.axis("off")
  plt.show()

## 6. Apparier les points caractéristiques

`feature.match_descriptors` compare les descripteurs de deux images, et associe chaque point de la première au point de la seconde dont le descripteur lui ressemble le plus. Vous en aurez besoin deux fois, ici et pour recoller deux images à l'exercice suivant : écrivez la fonction `apparier(image_1, image_2, max_ratio=1.0)`, qui détecte avec ORB les points caractéristiques des deux images et leurs descripteurs, comme à l'exercice précédent, apparie les descripteurs avec `feature.match_descriptors`, et renvoie les points de la première image, ceux de la seconde, et les paires.

Appliquez-la à `im` et `im_tournee`, dans `points_1`, `points_2` et `paires`, puis affichez les appariements entre les deux images avec `feature.plot_matched_features`. Les points des immeubles et des réverbères se retrouvent-ils au bon endroit dans l'image tournée ?

**Documentation utile**

- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.match_descriptors
- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.plot_matched_features

In [ ]:
def apparier(
  image_1: np.ndarray, image_2: np.ndarray, max_ratio: float = 1.0
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
  # Votre code ici
  ...


# Votre code ici

### Conseils supplémentaires

- Le code de détection de l'exercice précédent devient le début de la fonction, appliqué à `image_1` puis à `image_2`.
- `feature.match_descriptors(descripteurs_1, descripteurs_2, max_ratio=max_ratio)` renvoie un tableau à deux colonnes, une ligne par paire : l'indice du point dans `points_1`, puis celui de son correspondant dans `points_2`. Des descripteurs binaires, comme ceux d'ORB, se comparent par défaut par la distance de Hamming, le nombre de bits qui diffèrent. `max_ratio=1.0`, sa valeur par défaut, garde toutes les paires ; l'exercice suivant s'en sert pour écarter les paires ambiguës.
- Vous devez obtenir 320 paires entre `im` et `im_tournee`.
- `feature.plot_matched_features(image_1, image_2, keypoints0=..., keypoints1=..., matches=..., ax=ax)` trace les deux images côte à côte et relie les points de chaque paire ; `ax` vient de `fig, ax = plt.subplots(dpi=200)`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def apparier(
  image_1: np.ndarray, image_2: np.ndarray, max_ratio: float = 1.0
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
  detecteur = feature.ORB()
  detecteur.detect_and_extract(color.rgb2gray(image_1))
  points_1 = detecteur.keypoints
  descripteurs_1 = detecteur.descriptors

  detecteur.detect_and_extract(...)
  points_2 = ...
  descripteurs_2 = ...

  paires = feature.match_descriptors(..., ..., max_ratio=...)
  return points_1, points_2, paires


points_1, points_2, paires = apparier(..., ...)
print(len(paires), "paires")

fig, ax = plt.subplots(dpi=200)
feature.plot_matched_features(
  im, im_tournee, keypoints0=..., keypoints1=..., matches=paires, ax=ax
)
ax.axis("off")
plt.show()
```

### Solution

In [ ]:
def apparier(
  image_1: np.ndarray, image_2: np.ndarray, max_ratio: float = 1.0
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
  detecteur = feature.ORB()
  detecteur.detect_and_extract(color.rgb2gray(image_1))
  points_1 = detecteur.keypoints
  descripteurs_1 = detecteur.descriptors

  # Le détecteur remplace ses attributs à chaque appel.
  detecteur.detect_and_extract(color.rgb2gray(image_2))
  points_2 = detecteur.keypoints
  descripteurs_2 = detecteur.descriptors

  # Une ligne par paire : l'indice du point dans points_1, puis dans points_2.
  paires = feature.match_descriptors(
    descripteurs_1, descripteurs_2, max_ratio=max_ratio
  )
  return points_1, points_2, paires


points_1, points_2, paires = apparier(im, im_tournee)
print(len(paires), "paires")

In [ ]:
fig, ax = plt.subplots(dpi=200)
feature.plot_matched_features(
  im, im_tournee, keypoints0=points_1, keypoints1=points_2, matches=paires, ax=ax
)
ax.axis("off")
plt.show()

## 7. Recoller deux images qui se chevauchent

Deux photos prises l'une à côté de l'autre ont une partie commune : les points caractéristiques qu'elles partagent disent de combien l'une est décalée par rapport à l'autre, donc où les recoller. C'est le principe d'un panorama. Vous allez recoller deux morceaux de `im` qui se chevauchent, ce qui permet de vérifier le résultat : recollés, ils doivent redonner `im`.

### Découper l'image

Écrivez la fonction `decouper_en_deux(image, recouvrement)`, qui coupe `image` en deux morceaux ayant `recouvrement` colonnes en commun, au milieu de l'image : le morceau de gauche va du bord gauche de l'image à la fin de la zone commune, celui de droite du début de la zone commune au bord droit. Des tranches NumPy suffisent.

Appliquez-la à `im`, avec un recouvrement de 960 colonnes, la moitié de sa largeur, pour obtenir deux images `im_1` et `im_2`. Affichez leurs formes, puis les deux images côte à côte. Les deux morceaux doivent avoir la forme `(1247, 1440, 3)`.

In [ ]:
def decouper_en_deux(
  image: np.ndarray, recouvrement: int
) -> tuple[np.ndarray, np.ndarray]:
  # Votre code ici
  ...


# Votre code ici

#### Conseils supplémentaires

- `image.shape[1]` est la largeur de l'image. Centrée, la zone commune commence à la colonne `debut = (largeur - recouvrement) // 2`, et s'arrête avant la colonne `debut + recouvrement` : de 480 à 1440 pour `im`.
- `image[:, :fin]` garde les colonnes d'indice inférieur à `fin`, et `image[:, debut:]` celles à partir de `debut`, avec toutes les lignes et tous les canaux.
- `im_1` est `im` sans ses 480 dernières colonnes, `im_2` sans ses 480 premières.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def decouper_en_deux(
  image: np.ndarray, recouvrement: int
) -> tuple[np.ndarray, np.ndarray]:
  largeur = image.shape[1]
  debut = ...
  fin = debut + recouvrement
  return image[:, :fin], image[:, ...]


im_1, im_2 = decouper_en_deux(im, ...)
print(im_1.shape, im_2.shape)
cote_a_cote((im_1, im_2))
```

#### Solution

In [ ]:
def decouper_en_deux(
  image: np.ndarray, recouvrement: int
) -> tuple[np.ndarray, np.ndarray]:
  largeur = image.shape[1]
  # La zone commune, au milieu : de la colonne debut à la colonne fin, exclue.
  debut = (largeur - recouvrement) // 2
  fin = debut + recouvrement
  return image[:, :fin], image[:, debut:]


im_1, im_2 = decouper_en_deux(im, 960)
print(im_1.shape, im_2.shape)
cote_a_cote((im_1, im_2))

### Apparier les points des deux images

Appariez les points caractéristiques de `im_1` et de `im_2` avec votre fonction `apparier`, dans `points_1`, `points_2` et `paires`, en ne gardant que les paires sans ambiguïté grâce à son paramètre `max_ratio`. Affichez les appariements.

**Documentation utile**

- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.match_descriptors

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `max_ratio=0.8` écarte une paire quand le deuxième meilleur candidat ressemble presque autant au point que le meilleur, le rapport de leurs distances dépassant 0,8 : ces paires ambiguës sont souvent fausses.
- Vous devez obtenir 256 paires.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
points_1, points_2, paires = apparier(im_1, im_2, max_ratio=...)
print(len(paires), "paires")

fig, ax = plt.subplots(dpi=200)
feature.plot_matched_features(
  im_1, im_2, keypoints0=..., keypoints1=..., matches=paires, ax=ax
)
ax.axis("off")
plt.show()
```

#### Solution

In [ ]:
# Les paires ambiguës écartées : celles où le rapport des distances au meilleur
# candidat et au deuxième meilleur dépasse 0.8.
points_1, points_2, paires = apparier(im_1, im_2, max_ratio=0.8)
print(len(paires), "paires")

In [ ]:
fig, ax = plt.subplots(dpi=200)
feature.plot_matched_features(
  im_1, im_2, keypoints0=points_1, keypoints1=points_2, matches=paires, ax=ax
)
ax.axis("off")
plt.show()

### Taille de la zone commune

- Calculez l'écart moyen, en colonnes, entre les deux points de chaque paire, dans `decalage` : c'est le nombre de pixels dont `im_2` est décalée par rapport à `im_1`.
- Déduisez-en la largeur de la zone commune aux deux images, `taille_zone`, en pixels : la largeur de `im_1` moins ce décalage. Arrondissez-la à l'entier le plus proche : un nombre de pixels est entier, et une troncature par `int` donnerait une colonne de trop à recoller pour peu que le décalage dépasse un entier de quelques millièmes.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `points_1[paires[:, 0]]` et `points_2[paires[:, 1]]` mettent les points de chaque paire en face l'un de l'autre. La valeur absolue de leur différence, moyennée sur l'axe 0 (`.mean(axis=0)`), donne l'écart moyen en lignes, puis en colonnes : seul celui des colonnes compte.
- `round(x)` arrondit à l'entier le plus proche.
- Vous devez trouver un décalage d'environ 480 pixels, ce que `decouper_en_deux` a retiré de chaque côté, et une zone commune de 960 pixels, le recouvrement donné à `decouper_en_deux`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
_, decalage = np.abs(points_2[...] - points_1[...]).mean(axis=0)
taille_zone = round(...)
print(decalage, taille_zone)
```

#### Solution

In [ ]:
# L'écart moyen entre les points de chaque paire, en lignes et en colonnes.
_, decalage = np.abs(points_2[paires[:, 1]] - points_1[paires[:, 0]]).mean(axis=0)
# Arrondie, et non tronquée : int(1440 - 480.002) donnerait 959.
taille_zone = round(im_1.shape[1] - decalage)
print(decalage, taille_zone)

### Recoller les deux images

Retirez à `im_2` sa partie gauche, de la largeur de la zone commune, puis mettez le reste à droite de `im_1` avec `np.concatenate`, dans `im_fusion`. Affichez le résultat, et vérifiez qu'il redonne `im`.

**Documentation utile**

- https://numpy.org/doc/stable/reference/generated/numpy.concatenate.html
- https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `im_2[:, taille_zone:, :]` retire les `taille_zone` premières colonnes de `im_2`.
- `np.concatenate((a, b), axis=1)` met `b` à droite de `a` : l'axe 1 est celui des colonnes.
- `im_fusion` doit avoir la forme de `im`, `(1247, 1920, 3)`, et `np.array_equal(im_fusion, im)` doit renvoyer `True` : les deux images recollées redonnent exactement la photo de départ.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
im_fusion = np.concatenate((im_1, ...), axis=...)
print(im_fusion.shape, np.array_equal(im_fusion, im))

plt.figure(dpi=100)
plt.imshow(im_fusion)
plt.axis("off")
plt.show()
```

#### Solution

In [ ]:
im_fusion = np.concatenate((im_1, im_2[:, taille_zone:, :]), axis=1)
print(im_fusion.shape, np.array_equal(im_fusion, im))

plt.figure(dpi=100)
plt.imshow(im_fusion)
plt.axis("off")
plt.show()

### Pour aller plus loin : recoller en fondu (démonstration)

D'une photo à l'autre, la lumière change, et une coupure franche laisse une limite visible. La fonction `recoller_en_fondu` ci-dessous recolle plutôt les deux images par un fondu : au milieu de la zone commune, sur `2 * largeur_fondu` colonnes, le poids de `im_2` passe de 0 à 1 par une rampe linéaire, diffusée sur les lignes et les canaux par broadcasting, comme la rampe `alpha` de la « Fusion d'images » du TP Pillow et NumPy. La cellule l'essaie sur `im_2` assombrie, comme une seconde photo prise sous une autre lumière.

Exécutez-la, et regardez la limite entre les deux images : une marche nette avec la coupure franche, un passage progressif avec le fondu.

In [ ]:
def recoller_en_fondu(
  image_1: np.ndarray, image_2: np.ndarray, taille_zone: int, largeur_fondu: int = 30
) -> np.ndarray:
  # La zone commune : la fin de image_1, le début de image_2.
  fin_1 = image_1[:, -taille_zone:, :]
  debut_2 = image_2[:, :taille_zone, :]
  # Le poids de image_2 dans la zone commune : 0 à gauche, 1 à droite, et une
  # rampe linéaire sur 2 * largeur_fondu colonnes, au milieu.
  poids = np.zeros(taille_zone)
  milieu = taille_zone // 2
  poids[milieu:] = 1
  poids[milieu - largeur_fondu : milieu + largeur_fondu] = np.linspace(
    0, 1, 2 * largeur_fondu
  )
  # Un poids par colonne : la forme (1, taille_zone, 1) se diffuse sur les
  # lignes et les canaux.
  poids = poids[np.newaxis, :, np.newaxis]
  zone = fin_1 * (1 - poids) + debut_2 * poids
  return (
    np.concatenate(
      (image_1[:, :-taille_zone, :], zone, image_2[:, taille_zone:, :]), axis=1
    )
    .round()
    .astype(np.uint8)
  )


# im_2 assombrie, comme une seconde photo prise sous une autre lumière.
im_2_sombre = (im_2 * 0.7).astype(np.uint8)
coupure = np.concatenate((im_1, im_2_sombre[:, taille_zone:, :]), axis=1)
fondu = recoller_en_fondu(im_1, im_2_sombre, taille_zone, largeur_fondu=200)
cote_a_cote((coupure, fondu), sous_titres=("Coupure franche", "Fondu"), dpi=200)

## 8. Métriques de comparaison d'images

Une métrique de comparaison mesure à quel point deux images se ressemblent. Préparez trois versions modifiées de la photo :

- Convertissez `im` en flottants entre 0 et 1 dans `im_float`, avec `util.img_as_float` comme dans le cours, ou vous-même.
- Créez une image bruitée `im_bruitee` : ajoutez à `im_float` un bruit gaussien de même forme, tiré par `np.random.normal`, puis ramenez les valeurs entre 0 et 1 avec `np.clip`.
- Créez une image éclaircie `im_eclaircie` : `im_float` plus une constante, ramenée entre 0 et 1 de même.
- Créez l'image miroir `im_miroir`, le reflet gauche-droite de `im_float` (une symétrie d'axe vertical).

Comparez ensuite `im_float` à elle-même et à chacune des trois autres avec deux métriques, et affichez les quatre images côte à côte, avec les valeurs des métriques en sous-titre :

- `metrics.structural_similarity` (SSIM), conçue pour suivre notre perception de la ressemblance : entre -1 et 1, 1 pour deux images identiques ;
- `metrics.mean_squared_error` (MSE), l'erreur quadratique moyenne : de 0, pour deux images identiques, à l'infini.

Quelle image la MSE juge-t-elle la plus proche de l'originale ? Et le SSIM ? Lequel des deux s'accorde avec votre œil ?

**Documentation utile**

- https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.mean_squared_error
- https://numpy.org/doc/stable/reference/random/generated/numpy.random.normal.html
- https://numpy.org/doc/stable/reference/generated/numpy.clip.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `np.random.normal(0, 0.2, im_float.shape)` tire un bruit de moyenne 0 et d'écart type 0,2, de la forme de l'image ; `np.clip(tableau, 0, 1)` ramène à 0 ce qui est en dessous de 0, à 1 ce qui est au-dessus de 1.
- `im_float[:, ::-1]` inverse l'ordre des colonnes.
- `metrics.structural_similarity(a, b, channel_axis=2, data_range=1)` : `channel_axis` est l'axe des canaux, `data_range` l'écart entre la plus petite et la plus grande valeur possibles, 1 pour des flottants entre 0 et 1. `metrics.mean_squared_error(a, b)` n'a pas besoin de paramètre.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
im_float = util.img_as_float(im)
im_bruitee = np.clip(im_float + np.random.normal(0, 0.2, im_float.shape), 0, 1)
im_eclaircie = ...
im_miroir = ...

images = (im_float, im_bruitee, im_eclaircie, im_miroir)
sous_titres = []
for image in images:
  mse = metrics.mean_squared_error(im_float, image)
  ssim = ...
  sous_titres.append(f"MSE : {mse:.3f}, SSIM : {ssim:.2f}")
cote_a_cote(images, sous_titres=sous_titres, dpi=200)
```

### Solution

In [ ]:
im_float = util.img_as_float(im)
bruit = np.random.normal(0, 0.2, im_float.shape)
im_bruitee = np.clip(im_float + bruit, 0, 1)
im_eclaircie = np.clip(im_float + 0.2, 0, 1)
# Le reflet gauche-droite : les colonnes dans l'ordre inverse.
im_miroir = im_float[:, ::-1]

images = (im_float, im_bruitee, im_eclaircie, im_miroir)
sous_titres = []
for image in images:
  mse = metrics.mean_squared_error(im_float, image)
  ssim = metrics.structural_similarity(im_float, image, channel_axis=2, data_range=1)
  sous_titres.append(f"MSE : {mse:.3f}, SSIM : {ssim:.2f}")
cote_a_cote(images, sous_titres=sous_titres, dpi=200)

# Les trois images ont une MSE voisine, autour de 0,04, et la MSE juge même
# l'image bruitée (environ 0,035) un peu plus proche de l'originale que l'image
# éclaircie (environ 0,038). Le SSIM les sépare comme l'œil : l'image éclaircie,
# qui garde la structure de la photo, est la plus proche (environ 0,93), l'image
# bruitée la plus lointaine (environ 0,13), le miroir entre les deux (0,50).